# Extraction — image → identities

Produces `<RESULTS_ROOT>/<RUN_NAME>/extraction/`. No analysis-tunable
params; anything that depends on plot styling or geometry lives in
`notebooks/analysis.ipynb`.

In [ ]:
# Papermill-injected defaults (overridden by run_experiment.py).
RUN_NAME = "control_experiment"
RESULTS_ROOT = "../results"
STACK_PATH = "../data/Control-experiment/Grad LB+sucr-20-0.zvi  Ch0.tif"
FLUOR_PATH = "../data/Control-experiment/Grad LB+sucr-20-0.zvi  Ch1-BG.tif"
MODEL_TYPE = "cyto3"
DETECT_PARAMS = {
    "diameter": 32, "min_area": 300, "min_circularity": 0.7,
    "min_contrast": 1250, "exclude_edges": True,
    "gpu": True, "resample": False,
}
GATING_Z_THRESHOLD = 3.5
SEARCH_RANGE = 30.0
MEMORY = 3
MERGE_MAX_DISTANCE = 15.0
MERGE_MAX_GAP = 18
MIN_TRACK_DETECTIONS = 4
NUCLEUS_DIAMETER = 25
NUCLEUS_MIN_AREA = 100

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "../src")

from cell_analysis import (
    load_experiment, detect_cells_stack, run_frame_gating,
    run_tracking, filter_short_tracks, detect_nuclei_stack,
    plot_frame_preview, plot_detections, plot_channels_preview,
    plot_frame_gating, save_extraction, compute_provenance,
)

EXTRACTION_DIR = Path(RESULTS_ROOT) / RUN_NAME / "extraction"
EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extraction outputs will be written to {EXTRACTION_DIR}")

## 1. Load stacks

In [ ]:
phase_stack, fluor_stack = load_experiment(STACK_PATH, FLUOR_PATH)
plot_frame_preview(phase_stack)
plot_channels_preview(phase_stack, fluor_stack)

## 2. Cellpose detection (phase channel)

In [ ]:
centroids_all, label_stack = detect_cells_stack(
    phase_stack, model_type=MODEL_TYPE, **DETECT_PARAMS,
)
plot_detections(phase_stack[0], centroids_all[0])

## 3. Frame gating

In [ ]:
detections, bad_frames, diagnostics = run_frame_gating(
    label_stack, z_threshold=GATING_Z_THRESHOLD, results_dir=EXTRACTION_DIR,
)
plot_frame_gating(diagnostics, bad_frames)

## 4. Tracking + merging + minimum lifetime

In [ ]:
tracked, track_stats, merge_log = run_tracking(
    detections,
    search_range=SEARCH_RANGE, memory=MEMORY,
    merge_max_distance=MERGE_MAX_DISTANCE, merge_max_gap=MERGE_MAX_GAP,
)
tracked, track_stats = filter_short_tracks(
    tracked, track_stats, min_detections=MIN_TRACK_DETECTIONS,
)

## 5. Nucleus detection (fluor channel)

In [ ]:
nucleus_label_stack = detect_nuclei_stack(
    fluor_stack, diameter=NUCLEUS_DIAMETER, min_area=NUCLEUS_MIN_AREA,
    gpu=DETECT_PARAMS.get("gpu", False),
)

## 6. Persist extraction bundle

In [ ]:
import pandas as pd

REPO_ROOT = Path("..").resolve()

# Under papermill the runner injects absolute STACK_PATH/FLUOR_PATH.
# Interactively (cwd=notebooks/), STACK_PATH is like "../data/foo.tif" — a
# plain Path(...).resolve() gets the right absolute path in both cases.
stack_abs = Path(STACK_PATH).resolve()
fluor_abs = Path(FLUOR_PATH).resolve()

provenance = compute_provenance(
    {
        "STACK_PATH": STACK_PATH, "FLUOR_PATH": FLUOR_PATH,
        "MODEL_TYPE": MODEL_TYPE, "DETECT_PARAMS": DETECT_PARAMS,
        "GATING_Z_THRESHOLD": GATING_Z_THRESHOLD,
        "SEARCH_RANGE": SEARCH_RANGE, "MEMORY": MEMORY,
        "MERGE_MAX_DISTANCE": MERGE_MAX_DISTANCE,
        "MERGE_MAX_GAP": MERGE_MAX_GAP,
        "MIN_TRACK_DETECTIONS": MIN_TRACK_DETECTIONS,
        "NUCLEUS_DIAMETER": NUCLEUS_DIAMETER,
        "NUCLEUS_MIN_AREA": NUCLEUS_MIN_AREA,
    },
    stack_abs, fluor_abs,
    repo_root=REPO_ROOT,
)

# dropped_frames.csv already emitted by run_frame_gating when non-empty;
# ensure the file exists (empty is fine) for the extraction contract.
dropped_frames_path = EXTRACTION_DIR / "dropped_frames.csv"
if not dropped_frames_path.exists():
    pd.DataFrame({"frame": [], "reason": []}).to_csv(
        dropped_frames_path, index=False)

save_extraction(
    EXTRACTION_DIR,
    label_stack=label_stack,
    nucleus_label_stack=nucleus_label_stack,
    tracked=tracked, track_stats=track_stats,
    diagnostics=diagnostics, merge_log=merge_log,
    dropped_frames=pd.read_csv(dropped_frames_path),
    provenance=provenance,
)
print(f"Wrote extraction bundle to {EXTRACTION_DIR}")